## Extracción de Q (runoff) — TerraClimate

se extrae únicamente la variable `q` (escorrentía mensual en mm) de TerraClimate.
La escorrentía representa el agua que fluye sobre la superficie del suelo hacia los ríos —
una variable relevante porque está directamente relacionada con la dilución de nutrientes
y la carga de sedimentos en los cuerpos de agua.

Se mantiene separado del notebook principal de TerraClimate para poder re-ejecutarlo
sin repetir la extracción de las demás variables.

In [ ]:
!pip install uv
!uv pip install -r requirements.txt

In [ ]:
import snowflake
from snowflake.snowpark.context import get_active_session
session = get_active_session()

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import xarray as xr
from scipy.spatial import cKDTree
import pystac_client
import planetary_computer as pc
from tqdm import tqdm
import os

In [ ]:
# Las funciones son idénticas a las del notebook principal de TerraClimate
# Se replican aquí para que este notebook sea autosuficiente

def load_terraclimate_dataset():
    """Carga el dataset TerraClimate desde Planetary Computer en formato Zarr."""
    catalog = pystac_client.Client.open(
        "https://planetarycomputer.microsoft.com/api/stac/v1",
        modifier=pc.sign_inplace,
    )
    collection = catalog.get_collection("terraclimate")
    asset = collection.assets["zarr-abfs"]
    if "xarray:storage_options" in asset.extra_fields:
        ds = xr.open_zarr(
            asset.href,
            storage_options=asset.extra_fields["xarray:storage_options"],
            consolidated=True,
        )
    else:
        ds = xr.open_dataset(asset.href, **asset.extra_fields["xarray:open_kwargs"])
    return ds


def filterg(ds, var):
    """
    Extrae una variable de TerraClimate para el período 2011-2015
    y filtra los puntos de la grilla que están dentro de Sudáfrica.
    """
    ds_2011_2015 = ds[var].sel(time=slice("2011-01-01", "2015-12-31"))
    df_var_append = []
    for i in tqdm(range(len(ds_2011_2015.time))):
        df_var = ds_2011_2015.isel(time=i).to_dataframe().reset_index()
        df_var_filter = df_var[
            (df_var['lat'] > -35.18) & (df_var['lat'] < -21.72) &
            (df_var['lon'] > 14.97)  & (df_var['lon'] < 32.79)
        ]
        df_var_append.append(df_var_filter)
    df_var_final = pd.concat(df_var_append, ignore_index=True)
    df_var_final['time'] = df_var_final['time'].astype(str)
    df_var_final = df_var_final.rename(columns={"lat":"Latitude","lon":"Longitude","time":"Sample Date"})
    print(f"Filtering for {var} completed")
    return df_var_final


def assign_nearest_climate(sa_df, climate_df, var_name):
    """
    Asigna a cada muestra de calidad del agua el valor de la variable climática
    del punto de grilla más cercano en espacio y más próximo en tiempo.
    """
    sa_coords      = np.radians(sa_df[['Latitude', 'Longitude']].values)
    climate_coords = np.radians(climate_df[['Latitude', 'Longitude']].values)
    tree = cKDTree(climate_coords)
    dist, idx = tree.query(sa_coords, k=1)
    nearest_points = climate_df.iloc[idx].reset_index(drop=True)
    sa_df = sa_df.reset_index(drop=True)
    sa_df[['nearest_lat', 'nearest_lon']] = nearest_points[['Latitude', 'Longitude']]
    sa_df['Sample Date']      = pd.to_datetime(sa_df['Sample Date'],      dayfirst=True, errors='coerce')
    climate_df['Sample Date'] = pd.to_datetime(climate_df['Sample Date'], dayfirst=True, errors='coerce')
    climate_values = []
    for i in tqdm(range(len(sa_df)), desc=f"Mapping {var_name.upper()} values"):
        sample_date = sa_df.loc[i, 'Sample Date']
        nearest_lat = sa_df.loc[i, 'nearest_lat']
        nearest_lon = sa_df.loc[i, 'nearest_lon']
        subset = climate_df[
            (climate_df['Latitude']  == nearest_lat) &
            (climate_df['Longitude'] == nearest_lon)
        ]
        if subset.empty:
            climate_values.append(np.nan)
            continue
        nearest_idx = (subset['Sample Date'] - sample_date).abs().idxmin()
        climate_values.append(subset.loc[nearest_idx, var_name])
    return pd.DataFrame({var_name: climate_values})

## Extracción Q — entrenamiento

In [ ]:
# Cargar el dataset de calidad del agua — se usa solo para coordenadas y fechas
Water_Quality_df = pd.read_csv("water_quality_training_dataset.csv")
print(f"Registros de entrenamiento: {len(Water_Quality_df)}")

In [ ]:
Q_TRAIN_PATH = "tc_q_train.csv"

if os.path.exists(Q_TRAIN_PATH):
    # Si ya existe el archivo, no repetir la extracción (tarda varios minutos)
    print(f"{Q_TRAIN_PATH} ya existe, cargando...")
    q_train_df = pd.read_csv(Q_TRAIN_PATH)
else:
    print("Cargando dataset TerraClimate...")
    ds = load_terraclimate_dataset()
    print(f"Variables con 'q': {[v for v in ds.data_vars if 'q' in v.lower()]}")

    print("Filtrando q para la región y período...")
    tc_q = filterg(ds, 'q')

    print("Asignando valores de q a los puntos de entrenamiento...")
    q_train_df = assign_nearest_climate(Water_Quality_df, tc_q, 'q')
    q_train_df.to_csv(Q_TRAIN_PATH, index=False)
    print(f"Guardado: {Q_TRAIN_PATH}")

# Verificar alineación con el dataset original
assert len(q_train_df) == len(Water_Quality_df),     f"Desfase: q_train={len(q_train_df)}, WQ={len(Water_Quality_df)}"

print(f"Q entrenamiento: {len(q_train_df)} filas")
print(q_train_df['q'].describe().round(3))

In [ ]:
# Subir a Snowflake para que esté disponible en el notebook de modelado
q_train_df.to_csv(f"/tmp/{Q_TRAIN_PATH}", index=False)
session.sql(f"""
    PUT file:///tmp/{Q_TRAIN_PATH}
    'snow://workspace/USER$.PUBLIC."ChallengeEY_prueba2"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()
print(f"{Q_TRAIN_PATH} subido a Snowflake")

## Extracción Q — validación

In [ ]:
Validation_df = pd.read_csv("submission_template.csv")
print(f"Registros de validación: {len(Validation_df)}")

In [ ]:
Q_VAL_PATH = "tc_q_val.csv"

if os.path.exists(Q_VAL_PATH):
    print(f"{Q_VAL_PATH} ya existe, cargando...")
    q_val_df = pd.read_csv(Q_VAL_PATH)
else:
    # Si tc_q ya está en memoria (de la celda de entrenamiento), se reutiliza
    # Si no, se recarga el dataset y se vuelve a filtrar
    if 'tc_q' not in dir():
        print("Recalculando grilla TerraClimate Q...")
        if 'ds' not in dir():
            ds = load_terraclimate_dataset()
        tc_q = filterg(ds, 'q')

    print("Asignando valores de q a los puntos de validación...")
    q_val_df = assign_nearest_climate(Validation_df, tc_q, 'q')
    q_val_df.to_csv(Q_VAL_PATH, index=False)
    print(f"Guardado: {Q_VAL_PATH}")

assert len(q_val_df) == len(Validation_df),     f"Desfase: q_val={len(q_val_df)}, Validation={len(Validation_df)}"

print(f"Q validación: {len(q_val_df)} filas")
print(q_val_df['q'].describe().round(3))

In [ ]:
q_val_df.to_csv(f"/tmp/{Q_VAL_PATH}", index=False)
session.sql(f"""
    PUT file:///tmp/{Q_VAL_PATH}
    'snow://workspace/USER$.PUBLIC."ChallengeEY_prueba2"/versions/live/'
    AUTO_COMPRESS=FALSE
    OVERWRITE=TRUE
""").collect()
print(f"{Q_VAL_PATH} subido a Snowflake")